In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from crewai import Agent, Task, Crew, Process, LLM

import os
import asyncio
import nest_asyncio

nest_asyncio.apply()

model = os.getenv('MODEL_NAME', 'gpt-4o')

def build_email_crew(context: str, tone: str, recipient: str) -> str:
    llm = LLM(model=model, temperature=0.3)

    analyst = Agent(
        role="邮件上下文分析师",
        goal="理解邮件语境，提取关键信息，并定义邮件结构",
        backstory="你是一位专业的商务沟通分析师，擅长将复杂情况提炼为清晰的邮件需求。",
        llm=llm,
        verbose=True,
    )

    writer = Agent(
        role="专业邮件撰写人",
        goal="撰写清晰、简洁、高效的专业邮件",
        backstory="你是一位专业文案撰稿人，专注于撰写能获得回复的商务邮件。",
        llm=llm,
        verbose=True,
    )

    analyze_task = Task(
        description=f"""分析以下邮件需求：
        语境：{context}
        收件人：{recipient}
        期望语气：{tone}

        请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。""",
        agent=analyst,
        expected_output="结构化的邮件简报：目的、要点、行动号召和建议的邮件主题",
    )

    write_task = Task(
        description=f"""根据分析结果，撰写一封完整的商务邮件。
        语气：{tone}。收件人：{recipient}。
        请包含：邮件主题、称呼、正文段落、结尾敬语、签名占位符。
        正文请保持简洁——不超过200字。""",
        agent=writer,
        expected_output="可直接发送的完整格式邮件",
        context=[analyze_task],
    )

    crew = Crew(
        agents=[analyst, writer],
        tasks=[analyze_task, write_task],
        process=Process.sequential,
        verbose=True,
        tracing=True
    )

    result = asyncio.run(crew.kickoff_async())
    return str(result)

In [5]:
context = '跟进上周二的产品演示。他们表现出兴趣但尚未回复。'
tone = '专业且友好'
recipient = '潜在客户'

result = build_email_crew(context=context, tone=tone, recipient=recipient)

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 233e8a03-0d1a-4521-9343-33f0b4ba4836                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 分析以下邮件需求：                                                                                       │
│          语境：跟进上周二的产品演示。他们表现出兴趣但尚未回复。                                                 │
│          收件人：潜在客户                                                                                       │
│          期望语气：专业且友好                                                                                   │
│                                                                                                                 │
│          请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。                                              │
│  ID: d2013dc9-2623-419c-a682-1e279bda038c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 邮件上下文分析师                                                                                        │
│                                                                                                                 │
│  Task: 分析以下邮件需求：                                                                                       │
│          语境：跟进上周二的产品演示。他们表现出兴趣但尚未回复。                                                 │
│          收件人：潜在客户                                                                                       │
│          期望语气：专业且友好                                                                                   │
│                                                                                                                 │
│          请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 邮件上下文分析师                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **邮件简报**                                                                                                   │
│                                                                                                                 │
│  **目的：**                                                                                                     │
│  跟进上周二产品演示的后续进展，巩固客户已表现出的兴趣，解答其内部评估过程中可能产生的疑问，并温和推动销售流程   │
│  进入下一阶段（如方案细化、POC测试或商务洽谈）。                                                                │
│                                                                                                                 │
│  **要点：**                                                                                                     │
│  1.                                                                                                             │
│  **致谢与情境唤醒**：感谢对方团队参与上周二的演示，简要提及会议中其重点关注的业务场景或痛点，帮助对方快速回忆   │
│  沟通背景。                                                                                                     │
│  2.                                                                                                             │
│  **核心价值重申**：用1-2句话精炼重申产品如何针对性解决其需求，可附带一个关键指标、客户案例或演示中未及展开的亮  │
│  点作为补充支撑。                                                                                               │
│  3.                                                                                                             │
│  **主动提供支持**：表明理解对方可能正在进行内部讨论或流程审批，主动提供补充资料（如详细功能清单、ROI测算表、行  │
│  业案例）或安排技术/业务专家进行一对一答疑。                                                                    │
│  4.                                                                                                             │
│  **语气与节奏把控**：全程保持专业、友好、不具压迫感的沟通姿态，尊重对方的决策周期，避免频繁催促，体现长期合作   │
│  诚意。                                                                                                         │
│                                                                                                                 │
│  **行动号召（CTA）：**                                                                                          │
│  邀请客户在方便时直接回复本邮件分享初步反馈或疑问，或点击附带的日历链接预约一次15分钟的简短跟进通话，以便明确   │
│  下一步计划（如安排产品试用、提供定制化报价或对接技术团队）。                                                   │
│                                                                                                                 │
│  **建议的邮件主题：**                                                                                           │
│  跟进：上周二[产品名称]演示后续 / 关于演示内容的补充与下一步建议                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 专业邮件撰写人                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  主题：跟进：上周二[产品名称]演示后续与下一步建议                                                               │
│                                                                                                                 │
│  尊敬的[客户姓名]，您好！                                                                                       │
│                                                                                                                 │
│  感谢您团队参与上周二的产品演示。针对您重点关注的[核心业务场景]，我们的方案已助力同类客户实现[关键指标]。理解   │
│  贵司正进行内部评估，如需详细功能清单、ROI测算表或专家答疑，请随时告知。若您方便，欢迎直接回复分享初步反馈，或  │
│  点击[日历链接]预约15分钟简短通话，以便明确后续试用或报价安排。                                                 │
│                                                                                                                 │
│  祝商祺！                                                                                                       │
│                                                                                                                 │
│  [您的姓名]                                                                                                     │
│  [您的职位]                                                                                                     │
│  [公司名称]                                                                                                     │
│  [联系方式]                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 根据分析结果，撰写一封完整的商务邮件。                                                                   │
│          语气：专业且友好。收件人：潜在客户。                                                                   │
│          请包含：邮件主题、称呼、正文段落、结尾敬语、签名占位符。                                               │
│          正文请保持简洁——不超过200字。                                                                          │
│  Agent: 专业邮件撰写人                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

主题：跟进：上周二[产品名称]演示后续与下一步建议

尊敬的[客户姓名]，您好！

感谢您团队参与上周二的产品演示。针对您重点关注的[核心业务场景]，我们的方案已助力同类客户实现[关键指标]。理解贵司正进行内部评估，如需详细功能清单、ROI测算表或专家答疑，请随时告知。若您方便，欢迎直接回复分享初步反馈，或点击[日历链接]预约15分钟简短通话，以便明确后续试用或报价安排。

祝商祺！

[您的姓名]
[您的职位]
[公司名称]
[联系方式]


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 233e8a03-0d1a-4521-9343-33f0b4ba4836                                                                       │
│  Final Output: 主题：跟进：上周二[产品名称]演示后续与下一步建议                                                 │
│                                                                                                                 │
│  尊敬的[客户姓名]，您好！                                                                                       │
│                                                                                                                 │
│  感谢您团队参与上周二的产品演示。针对您重点关注的[核心业务场景]，我们的方案已助力同类客户实现[关键指标]。理解   │
│  贵司正进行内部评估，如需详细功能清单、ROI测算表或专家答疑，请随时告知。若您方便，欢迎直接回复分享初步反馈，或  │
│  点击[日历链接]预约15分钟简短通话，以便明确后续试用或报价安排。                                                 │
│                                                                                                                 │
│  祝商祺！                                                                                                       │
│                                                                                                                 │
│  [您的姓名]                                                                                                     │
│  [您的职位]                                                                                                     │
│  [公司名称]                                                                                                     │
│  [联系方式]                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

model = os.getenv('MODEL_NAME', 'gpt-4o')
llm = ChatOpenAI(model=model, temperature=0.3, extra_body={"enable_thinking": False})

class EmailState(TypedDict):
    context: str
    tone: str
    recipient: str
    analysis: str
    email: str

def analyze_node(state: EmailState) -> EmailState:
    messages = [
        SystemMessage(content="你是一位专业的商务沟通分析师。"),
        HumanMessage(content=f"""分析以下邮件需求：
        语境: {state['context']}
        收件人: {state['recipient']}
        期望语气: {state['tone']}
        请提取: 目的、要点、CTA、建议主题"""),
    ]
    result = llm.invoke(messages)
    return {
        "context": state["context"],
        "tone": state["tone"],
        "recipient": state["recipient"],
        "analysis": result.content,
        "email": "",
    }

def write_node(state: EmailState) -> EmailState:
    messages = [
        SystemMessage(content="你是一位专业的商务邮件撰稿人。"),
        HumanMessage(content=f"""根据分析结果撰写一封完整的商务邮件：
        分析结果: {state['analysis']}
        语气: {state['tone']}
        收件人: {state['recipient']}
        请包含: 邮件主题、称呼、正文段落、结尾敬语、签名占位符
        正文不超过200字"""),
    ]
    result = llm.invoke(messages)
    return {
        "context": state["context"],
        "tone": state["tone"],
        "recipient": state["recipient"],
        "analysis": state["analysis"],
        "email": result.content,
    }

# 构建图
workflow = StateGraph(EmailState)
workflow.add_node("analyze", analyze_node)
workflow.add_node("write", write_node)
workflow.set_entry_point("analyze")
workflow.add_edge("analyze", "write")
workflow.add_edge("write", END)
app = workflow.compile()


input_state: EmailState = {
    "context": context,
    "tone": tone,
    "recipient": recipient,
    "analysis": "",
    "email": ""
}
# 执行
for chunk in app.stream(input=input_state, stream_mode="messages"):
    msg, meta = chunk
    if hasattr(msg, 'content'):
        print(getattr(msg, 'content'), end='', flush=False)


基于您提供的语境和约束条件，以下是对该跟进邮件需求的详细分析及建议：

### 1. 目的 (Purpose)
*   **核心目标**：重新激活对话，确认潜在客户对上周二产品演示的兴趣程度。
*   **次要目标**：消除客户可能存在的疑虑或障碍，推动销售流程进入下一阶段（如安排下一步会议、发送报价或试用申请）。
*   **关系维护**：在保持专业度的同时，通过友好的语气强化合作关系，避免给客户造成“催促”的压力感。

### 2. 要点 (Key Points)
邮件正文应包含以下关键信息点，逻辑需清晰简洁：
*   **回顾与感谢**：简要提及上周二的演示，并感谢对方拨冗参加及表现出的兴趣。
*   **价值重申**：用一句话概括演示中解决的核心痛点或带来的最大价值（例如：“正如我们讨论的，我们的方案能帮助您将效率提升X%”），以唤起记忆。
*   **提供额外支持**：询问是否有未解答的问题，或主动提供补充材料（如案例研究、详细规格表、竞品对比等），体现服务意识。
*   **低压力跟进**：表明理解对方忙碌，此次联系仅为确保信息同步，而非强行推销。

### 3. CTA (Call to Action / 行动号召)
CTA 必须明确、单一且易于执行。鉴于客户尚未回复，建议采用**低门槛**的行动号召：
*   **首选 CTA**：询问是否方便进行一个简短的 10-15 分钟电话沟通，以解答任何遗留问题。
    *   *示例话术*：“如果您有任何疑问，或者想深入探讨某个细节，我可以在本周三或周四安排一个简短的电话会议。您看哪个时间比较方便？”
*   **备选 CTA**（如果客户更倾向于异步沟通）：请求对方确认是否需要发送更多特定资料。
    *   *示例话术*：“或者，如果您目前还在内部评估，我可以先发送一份详细的 ROI 分析报告供您参考。请告诉我您的偏好。”

### 4. 建议主题 (Suggested Subject Lines)
主题行需要既专业又具有亲和力，避免过于激进或模糊。以下是几个不同侧重点的建议：

*   **侧重回顾与价值**：
    *   `关于周二演示的后续：[客户公司名称] 的效率提升方案`
    *   `跟进：上周二 [产品名称] 演示的关键要点`

*   **侧重提供帮助/低压力**：
    